# Has AI Become Dumber?

### A reproducible investigation of public evaluations, September 2025 to September 2026

**Status:** starter notebook. There are no downloaded datasets or empirical findings here yet. Run the setup cells, then add documented source snapshots before drawing conclusions.

**Question:** Across comparable evaluations, did measured AI capability decline, increase, or move differently by domain? Product experience and user preference are separate questions that need their own evidence.

## 1. Design and guardrails

- Keep *model release date*, *evaluation date*, and *leaderboard snapshot date* distinct.
- Compare scores only within a stable benchmark version, task suite, methodology, and scoring rule.
- A leaderboard is a selected set of submissions, not a random sample of all models or users.
- Random sampling is for reproducible exploration **within a defined sampling frame**. It does not remove selection bias in that frame.
- A bootstrap interval describes sampling uncertainty under its assumptions; it does not correct benchmark drift, correlated models, or measurement bias.
- Keep METR minutes, LiveBench scores, and SWE-bench percentages separate.

Before publication, record the exact source revision and cleaning rules in `data/SOURCES.md`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
print(f"Project root: {ROOT.resolve()}")
print(f"NumPy {np.__version__}; pandas {pd.__version__}")

## 2. Source inventory

The three initial source candidates are METR, LiveBench, and SWE-bench. Their repository URLs and expected local schemas are in `data/SOURCES.md`. The files below are user-prepared normalized extracts; the notebook does not silently download changing leaderboards.

In [ ]:
expected_files = {
    "METR": PROCESSED / "metr_horizons.csv",
    "LiveBench": PROCESSED / "livebench_scores.csv",
    "SWE-bench Verified": PROCESSED / "swebench_verified.csv",
}
inventory = pd.DataFrame(
    [(name, str(path.relative_to(ROOT)), path.exists()) for name, path in expected_files.items()],
    columns=["source", "local_file", "available"],
)
print(inventory.to_string(index=False))

## 3. Reproducible sampling and uncertainty helpers

These helpers are ready for cleaned tables. Choose the population, strata, and unit of observation before calling them. Do not sample task runs and then report them as independent models.

In [ ]:
def stratified_sample(frame, strata, n_per_group, seed=RANDOM_SEED):
    """Sample up to n_per_group rows per observed stratum, without replacement."""
    if n_per_group < 1:
        raise ValueError("n_per_group must be positive")
    generator = np.random.default_rng(seed)
    parts = []
    for _, group in frame.groupby(strata, dropna=False, sort=True):
        draw_seed = int(generator.integers(0, 2**32 - 1))
        parts.append(group.sample(n=min(n_per_group, len(group)), random_state=draw_seed))
    if not parts:
        return frame.iloc[0:0].copy()
    return pd.concat(parts).sort_index()

def bootstrap_difference(a, b, statistic=np.median, iterations=5000, seed=RANDOM_SEED):
    """Percentile interval for statistic(b) - statistic(a); assumes independent rows."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0 or iterations < 1:
        raise ValueError("Both groups and a positive iteration count are required")
    generator = np.random.default_rng(seed)
    differences = np.empty(iterations)
    for i in range(iterations):
        differences[i] = statistic(generator.choice(b, len(b), replace=True)) - statistic(
            generator.choice(a, len(a), replace=True)
        )
    return {
        "observed_difference": float(statistic(b) - statistic(a)),
        "ci_low": float(np.percentile(differences, 2.5)),
        "ci_high": float(np.percentile(differences, 97.5)),
    }

## 4. METR: autonomous task horizon

Supply `data/processed/metr_horizons.csv` with `model,release_date,eval_date,methodology,p50_minutes`. Record the upstream report version and conversion to minutes. The scatter below is descriptive; distinct methodologies must be analyzed separately.

In [ ]:
metr_path = expected_files["METR"]
if metr_path.exists():
    metr = pd.read_csv(metr_path, parse_dates=["release_date", "eval_date"])
    required = {"model", "release_date", "eval_date", "methodology", "p50_minutes"}
    missing = required - set(metr.columns)
    if missing:
        raise ValueError(f"METR extract lacks columns: {sorted(missing)}")
    metr["p50_minutes"] = pd.to_numeric(metr["p50_minutes"], errors="coerce")
    valid = metr.dropna(subset=["release_date", "p50_minutes"])
    valid = valid.loc[valid["p50_minutes"] > 0]
    print(f"Rows: {len(metr)}; valid positive horizons: {len(valid)}")
    for methodology, subset in valid.groupby("methodology"):
        plt.scatter(subset["release_date"], subset["p50_minutes"], label=str(methodology), alpha=.75)
    if not valid.empty:
        plt.yscale("log")
        plt.axvline(pd.Timestamp("2025-09-01"), color="gray", linestyle="--", linewidth=1)
        plt.xlabel("Model release date")
        plt.ylabel("Estimated p50 horizon (minutes; log scale)")
        plt.title("METR task horizon by release date and methodology")
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print("METR extract missing. See data/SOURCES.md before preparing it.")

## 5. LiveBench: performance by category

Planned input: `livebench_scores.csv`. Inspect the distribution and comparable model cohorts **within each benchmark release**. A newer test set can change difficulty, so a raw change between releases is not automatically a capability change.

In [ ]:
livebench_path = expected_files["LiveBench"]
if livebench_path.exists():
    livebench = pd.read_csv(livebench_path, parse_dates=["release_date", "snapshot_date"])
    print(livebench.groupby(["benchmark_release", "category"], dropna=False)["score"].agg(["count", "median"]))
else:
    print("LiveBench extract missing; category analysis pending source acquisition.")

## 6. SWE-bench Verified: coding systems

Planned input: `swebench_verified.csv`. Filter to Verified and a stable benchmark version before interpreting changes. Submission dates are not model release dates, and the evaluated system includes scaffolding and tools.

In [ ]:
swe_path = expected_files["SWE-bench Verified"]
if swe_path.exists():
    swe = pd.read_csv(swe_path, parse_dates=["submission_date"])
    print(swe.groupby("benchmark_version", dropna=False)["resolved_pct"].agg(["count", "median"]))
else:
    print("SWE-bench Verified extract missing; coding analysis pending source acquisition.")

## 7. September 2025 vs September 2026

**To decide after inspecting counts:** comparison windows, stable benchmark versions, model-family matching, unit of analysis, and statistic. A Sep 2025 model can appear in a Sep 2026 leaderboard, so snapshot-only comparisons need care. Report sample sizes and missingness for each window. Do not fill in a summary table before the sources are harmonized.

In [ ]:
comparison_windows = {
    "earlier": (pd.Timestamp("2025-09-01"), pd.Timestamp("2025-10-31")),
    "later": (pd.Timestamp("2026-08-01"), pd.Timestamp("2026-09-30")),
}
print(pd.DataFrame(comparison_windows, index=["start", "end"]).T)

## 8. Human preference and everyday reliability

Historical human-preference snapshots require a documented data source and comparable rating scale. Public benchmark scores alone cannot show whether an AI product feels more reliable. For a reliability claim, obtain repeated trials on fixed tasks with stable prompts, settings, tools, and success criteria; measure variance or failure rate at the task level. This section is a research gap, not a result.

## 9. Limitations and conclusion (complete after analysis)

Track benchmark saturation, changing task sets, selection of leaderboard submissions, correlated model families, agent scaffolding, missing evaluations, model aliases, and uncertainty. State which claims the available data can and cannot support. Then write a Medium post using only figures produced from documented source snapshots.